# 편향과 분산 실습

**Bias-Variance Tradeoff · 편향-분산 절충**

모델이 단순해서 생기는 오차와 데이터에 민감해서 생기는 오차 사이의 절충 관계.

소재 분야에서 이해하기: 다항식 차수를 올리면 편향은 줄지만 분산이 커진다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 용어집](https://scikit-learn.org/stable/glossary.html)

## 1. 편향과 분산을 직접 분해

같은 문제를 서로 다른 학습 데이터로 여러 번 학습해, 예측의 평균 오차와 흔들림을 나눠봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

grid = np.linspace(0, 1, 120)
truth = np.sin(2 * np.pi * grid)

def repeated_fits(degree, repeats=60, n=25, noise=0.25):
    curves = []
    for repeat in range(repeats):
        local = np.random.default_rng(repeat)
        x = local.uniform(0, 1, n)
        y = np.sin(2 * np.pi * x) + local.normal(0, noise, n)
        model = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x[:, None], y)
        curves.append(model.predict(grid[:, None]))
    return np.array(curves)

for degree in (1, 3, 12):
    curves = repeated_fits(degree)
    bias2 = np.mean((curves.mean(0) - truth) ** 2)
    variance = np.mean(curves.var(0))
    print('차수 %2d  편향^2 %.4f  분산 %.4f  합 %.4f' % (degree, bias2, variance, bias2 + variance))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for axis, degree in zip(axes, (1, 3, 12)):
    curves = repeated_fits(degree)
    for curve in curves[:25]:
        axis.plot(grid, curve, color='steelblue', alpha=0.25, lw=1)
    axis.plot(grid, truth, 'k--', lw=2)
    axis.set_title('degree %d' % degree); axis.set_ylim(-2.5, 2.5)
plt.tight_layout(); plt.show()

## 2. 해석

차수 1은 참 함수를 못 따라가 **편향**이 큽니다. 차수 12는 학습 데이터마다 크게 달라져 **분산**이
큽니다. 중간 차수에서 합이 최소가 됩니다. 데이터를 늘리면 분산이 줄어 더 복잡한 모델을 쓸 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#bias-variance)을 여세요.